In [20]:
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display
import tqdm as tqdm
from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cdist
from scipy.spatial.distance import pdist
import os

In [ ]:
df = pd.read_csv("/home/guilherme_monteiro/projetos/tcc/data/metadata/video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "/home/guilherme_monteiro/projetos/tcc/data/videos/" + x)
df_true = df[df['Video Ground Truth'] == "Real"].reset_index(drop=True)
df_fake = df[df['Video Ground Truth'] == "Fake"].reset_index(drop=True)

def metadata_path_for_video(video_path):
    video_stem = os.path.splitext(os.path.basename(video_path))[0]
    return os.path.join(
        "/home/guilherme_monteiro/projetos/tcc/data/metadata",
        f"{video_stem}_meta.json"
    )

df['metadata_path'] = df['video_path'].apply(metadata_path_for_video)
df['has_metadata'] = df['metadata_path'].apply(os.path.exists)
df_meta = df[df['has_metadata']].reset_index(drop=True)

print(f"Vídeos no CSV: {len(df)}")
print(f"Vídeos com metadata disponível: {len(df_meta)}")
print(df_meta['Video Ground Truth'].value_counts())


# Funções básicas

In [ ]:
def get_video_frame_count(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frame_count


def sample_frame_indices(frame_count, max_frames=None):
    if frame_count <= 0:
        return np.array([], dtype=int)

    if max_frames is None or frame_count <= max_frames:
        return np.arange(frame_count, dtype=int)

    return np.linspace(0, frame_count - 1, int(max_frames)).astype(int)


def load_video_frames(video_path, max_frames=None, return_indices=False):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = sample_frame_indices(frame_count, max_frames)

    frames = []
    used_indices = []

    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()

        if not ret:
            continue

        frames.append(frame)
        used_indices.append(int(frame_idx))

    cap.release()

    frames = np.array(frames)

    if return_indices:
        return frames, np.array(used_indices, dtype=int), frame_count

    return frames


def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]

    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)

    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return frame_resized, scale


def load_metadata(video_path):
    full_path = metadata_path_for_video(video_path)

    with open(full_path, "r") as f:
        return json.load(f)


def metadata_index_for_frame(frame_idx, frame_count, metadata_len):
    if metadata_len <= 0:
        return None

    if metadata_len >= frame_count - 3:
        return min(int(frame_idx), metadata_len - 1)

    metadata_frame_indices = sample_frame_indices(frame_count, metadata_len)
    return int(np.argmin(np.abs(metadata_frame_indices - int(frame_idx))))


def scale_bbox(bbox, scale):
    return [int(round(x * scale)) for x in bbox]


def clip_bbox(bbox, width, height, min_size=4):
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]

    x1 = max(0, min(width - 1, x1))
    x2 = max(0, min(width, x2))
    y1 = max(0, min(height - 1, y1))
    y2 = max(0, min(height, y2))

    if x2 <= x1:
        x2 = min(width, x1 + min_size)
    if y2 <= y1:
        y2 = min(height, y1 + min_size)

    return [x1, y1, x2, y2]


def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = clip_bbox(bbox, w, h)

    bw = x2 - x1
    bh = y2 - y1

    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    border_mask = expanded_mask - face_mask
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox": (x1, y1, x2, y2),
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }


## Função para visualização

In [23]:
def draw_text_block(img, text_lines, x=10, y=20, line_height=18):
    for i, line in enumerate(text_lines):
        cv2.putText(
            img,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    return img

# Desenha a caixa delimitadora no frame
def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()

    face_mask = regions["face"]
    border_mask = regions["border"]
    background_mask = regions["background"]

    overlay[face_mask == 1] = (0, 255, 0)       # verde
    overlay[border_mask == 1] = (0, 0, 255)     # vermelho
    overlay[background_mask == 1] = (255, 0, 0) # azul

    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

# SIFT

In [ ]:
def create_sift():
    return cv2.SIFT_create(nfeatures=500)


def _sample_descriptors(descriptors, max_descriptors=80):
    if descriptors is None or len(descriptors) <= max_descriptors:
        return descriptors

    idx = np.linspace(0, len(descriptors) - 1, max_descriptors).astype(int)
    return descriptors[idx]


def _safe_cosine_distances(a, b=None):
    if a is None or len(a) == 0:
        return np.array([])

    if b is None:
        if len(a) < 2:
            return np.array([])
        dists = pdist(a, metric="cosine")
    else:
        if b is None or len(b) == 0:
            return np.array([])
        dists = cdist(a, b, metric="cosine").ravel()

    return np.nan_to_num(dists, nan=1.0, posinf=1.0, neginf=1.0)


def compute_sift_region(gray, mask, sift):
    mask_uint8 = (mask * 255).astype(np.uint8)
    area = float(np.sum(mask))

    if area < 25:
        return None

    keypoints, descriptors = sift.detectAndCompute(gray, mask_uint8)
    kp_count = 0 if keypoints is None else len(keypoints)

    if descriptors is None or kp_count == 0:
        return {
            "kp_count": 0.0,
            "kp_density": 0.0,
            "desc_entropy_norm": 0.0,
            "desc_self_similarity": 0.0,
            "desc_mean": 0.0,
            "desc_std": 0.0,
            "response_mean": 0.0,
            "response_std": 0.0,
            "has_kp": 0.0,
            "descriptors": None,
        }

    descriptors = descriptors.astype(np.float32)
    sampled_descriptors = _sample_descriptors(descriptors)

    kp_density = float(kp_count / (area + 1e-6))

    desc_flat = descriptors.ravel()
    hist, _ = np.histogram(desc_flat, bins=64, range=(0, 255), density=False)
    hist = hist.astype(np.float64)
    hist = hist / (np.sum(hist) + 1e-12)
    desc_entropy = float(-np.sum(hist * np.log(hist + 1e-12)) / np.log(len(hist)))

    if len(sampled_descriptors) > 1:
        dists = _safe_cosine_distances(sampled_descriptors)
        desc_self_sim = float(1 - np.mean(dists)) if len(dists) else 0.0
    else:
        desc_self_sim = 1.0

    responses = np.array([kp.response for kp in keypoints], dtype=np.float32)

    return {
        "kp_count": float(kp_count),
        "kp_density": kp_density,
        "desc_entropy_norm": desc_entropy,
        "desc_self_similarity": desc_self_sim,
        "desc_mean": float(np.mean(descriptors)),
        "desc_std": float(np.std(descriptors)),
        "response_mean": float(np.mean(responses)),
        "response_std": float(np.std(responses)),
        "has_kp": 1.0,
        "descriptors": sampled_descriptors,
    }


In [ ]:
def sift_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    comparable_metrics = [
        "kp_count",
        "kp_density",
        "desc_entropy_norm",
        "desc_self_similarity",
        "desc_mean",
        "desc_std",
        "response_mean",
        "response_std",
        "has_kp",
    ]

    features = {
        f"{prefix}_sift_{metric}_diff": abs(f1[metric] - f2[metric])
        for metric in comparable_metrics
    }

    if f1["descriptors"] is not None and f2["descriptors"] is not None:
        dists = _safe_cosine_distances(f1["descriptors"], f2["descriptors"])
        features[f"{prefix}_sift_desc_dist"] = float(np.mean(dists)) if len(dists) else 1.0
    else:
        features[f"{prefix}_sift_desc_dist"] = 1.0

    return features


In [ ]:
def compute_sift_metrics(frame, bbox):
    frame_std, scale = standardize_frame(frame)
    bbox_scaled = scale_bbox(bbox, scale)
    bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])

    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)
    regions = create_face_regions(frame_std, bbox_scaled)
    sift = create_sift()

    face = compute_sift_region(gray, regions["face"], sift)
    border = compute_sift_region(gray, regions["border"], sift)
    bg = compute_sift_region(gray, regions["background"], sift)

    features = {}

    for region_name, region_features in [("face", face), ("border", border), ("background", bg)]:
        if region_features is None:
            continue

        for metric_name, metric_value in region_features.items():
            if metric_name == "descriptors":
                continue
            features[f"sift_{region_name}_{metric_name}"] = metric_value

    features.update(sift_region_differences(face, bg, "face_bg"))
    features.update(sift_region_differences(face, border, "face_border"))
    features.update(sift_region_differences(border, bg, "border_bg"))

    return features


## Video para debbug

In [ ]:
def sift_visual(gray, regions, sift):
    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

    colors = {
        "face": (0, 255, 0),
        "border": (0, 0, 255),
        "background": (255, 0, 0),
    }

    for region_name in ["face", "border", "background"]:
        mask_uint8 = (regions[region_name] * 255).astype(np.uint8)
        keypoints, _ = sift.detectAndCompute(gray, mask_uint8)

        for kp in keypoints or []:
            px, py = int(kp.pt[0]), int(kp.pt[1])
            cv2.circle(vis, (px, py), 1, colors[region_name], -1)

    return vis


def debug_sift_video(video_path, max_frames=120, interval=60):
    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)
    debug_frames = []
    sift = create_sift()

    print("\nProcessando SIFT...")

    for i, frame in enumerate(tqdm.tqdm(frames)):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        frame_std, scale = standardize_frame(frame)
        gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

        bbox = metadata[metadata_idx]["bbox"]
        bbox_scaled = scale_bbox(bbox, scale)
        bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])
        regions = create_face_regions(frame_std, bbox_scaled)

        features = compute_sift_metrics(frame, bbox)
        sift_map = sift_visual(gray, regions, sift)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {frame_idx} | Meta: {metadata_idx}",
            f"Face KP Count: {features.get('sift_face_kp_count', 0):.0f}",
            f"Face KP Density: {features.get('sift_face_kp_density', 0):.4f}",
            f"Face Entropy: {features.get('sift_face_desc_entropy_norm', 0):.3f}",
            f"Face SelfSim: {features.get('sift_face_desc_self_similarity', 0):.3f}",
            f"Face-BG DescDist: {features.get('face_bg_sift_desc_dist', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(sift_map, cv2.COLOR_BGR2RGB),
        ])

        debug_frames.append(combined)

    if not debug_frames:
        raise ValueError("Nenhum frame válido para visualização")

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")
    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(fig, update, frames=len(debug_frames), interval=interval, blit=True)

    plt.close(fig)
    display(HTML(ani.to_jshtml()))


In [28]:
# debug_sift_video(df_true['video_path'].iloc[0], max_frames=300)

In [29]:
# debug_sift_video(df_fake['video_path'].iloc[0], max_frames=300)

# Patch Similarity

In [ ]:
def extract_patches(gray, mask, patch_size=16, stride=8, min_mask_ratio=0.6, max_patches=80):
    patches = []
    h, w = gray.shape

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch_mask = mask[y:y + patch_size, x:x + patch_size]

            if np.mean(patch_mask) < min_mask_ratio:
                continue

            patch = gray[y:y + patch_size, x:x + patch_size].astype(np.float32)
            patch = patch - np.mean(patch)
            std = np.std(patch)

            if std < 1e-6:
                continue

            patch = patch / std
            patches.append(patch.flatten())

    if len(patches) < 2:
        return None

    patches = np.array(patches, dtype=np.float32)

    if len(patches) > max_patches:
        idx = np.linspace(0, len(patches) - 1, max_patches).astype(int)
        patches = patches[idx]

    return patches


In [ ]:
def compute_patch_similarity(patches):
    if patches is None or len(patches) < 2:
        return None

    dists = _safe_cosine_distances(patches)

    if len(dists) == 0:
        return None

    sims = 1 - dists

    return {
        "sim_mean": float(np.mean(sims)),
        "sim_std": float(np.std(sims)),
        "sim_median": float(np.median(sims)),
        "sim_p95": float(np.percentile(sims, 95)),
        "patch_count": float(len(patches)),
    }


def compute_patch_region(gray, mask):
    patches = extract_patches(gray, mask)

    if patches is None:
        return None

    return compute_patch_similarity(patches)


def patch_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    comparable_metrics = ["sim_mean", "sim_std", "sim_median", "sim_p95", "patch_count"]

    return {
        f"{prefix}_patch_{metric}_diff": abs(f1[metric] - f2[metric])
        for metric in comparable_metrics
    }


In [ ]:
def compute_patch_metrics(frame, bbox):
    frame_std, scale = standardize_frame(frame)
    bbox_scaled = scale_bbox(bbox, scale)
    bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])

    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)
    regions = create_face_regions(frame_std, bbox_scaled)

    face = compute_patch_region(gray, regions["face"])
    border = compute_patch_region(gray, regions["border"])
    bg = compute_patch_region(gray, regions["background"])

    features = {}

    for region_name, region_features in [("face", face), ("border", border), ("background", bg)]:
        if region_features is None:
            continue

        for metric_name, metric_value in region_features.items():
            features[f"patch_{region_name}_{metric_name}"] = metric_value

    features.update(patch_region_differences(face, bg, "face_bg"))
    features.update(patch_region_differences(face, border, "face_border"))
    features.update(patch_region_differences(border, bg, "border_bg"))

    return features


## Video para debbug

In [ ]:
def patch_similarity_visual(gray, mask, patch_size=16, stride=8):
    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    h, w = gray.shape

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch_mask = mask[y:y + patch_size, x:x + patch_size]

            if np.mean(patch_mask) < 0.6:
                continue

            patch = gray[y:y + patch_size, x:x + patch_size]
            var = np.var(patch)
            color = int(np.clip(var / 50, 0, 255))
            cv2.rectangle(vis, (x, y), (x + patch_size, y + patch_size), (color, 255 - color, 0), 1)

    return vis


def debug_patch_video(video_path, max_frames=120, interval=60):
    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)
    debug_frames = []

    print("\nProcessando Patch Similarity...")

    for i, frame in enumerate(tqdm.tqdm(frames)):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        frame_std, scale = standardize_frame(frame)
        gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

        bbox = metadata[metadata_idx]["bbox"]
        bbox_scaled = scale_bbox(bbox, scale)
        bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])
        regions = create_face_regions(frame_std, bbox_scaled)

        features = compute_patch_metrics(frame, bbox)
        patch_vis = patch_similarity_visual(gray, regions["face"])

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {frame_idx} | Meta: {metadata_idx}",
            f"Face Sim Mean: {features.get('patch_face_sim_mean', 0):.3f}",
            f"Face Sim Std: {features.get('patch_face_sim_std', 0):.3f}",
            f"Face Patch Count: {features.get('patch_face_patch_count', 0):.0f}",
            f"Face-Border SimDiff: {features.get('face_border_patch_sim_mean_diff', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(patch_vis, cv2.COLOR_BGR2RGB),
        ])

        debug_frames.append(combined)

    if not debug_frames:
        raise ValueError("Nenhum frame válido para visualização")

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")
    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(fig, update, frames=len(debug_frames), interval=interval, blit=True)

    plt.close(fig)
    display(HTML(ani.to_jshtml()))


> Testar metodo sem patch similarity, explorar a metrica mais a fundo no futuro

## Todas as metricas

In [ ]:
def all_metrics(video_path, max_frames=500, label=None, include_patch=True):
    video_name = os.path.basename(video_path)
    print(f"\nExtraindo métricas do vídeo: {video_name}")

    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)
    all_features = []

    for i, frame in enumerate(frames):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        bbox = metadata[metadata_idx]["bbox"]

        features_sift = compute_sift_metrics(frame, bbox)
        features_patch = compute_patch_metrics(frame, bbox) if include_patch else {}

        features = {**features_sift, **features_patch}
        features["frame"] = frame_idx
        features["metadata_idx"] = metadata_idx

        all_features.append(features)

    metrics_df = pd.DataFrame(all_features)
    metrics_df.insert(0, "video_name", video_name)

    if label is not None:
        metrics_df.insert(1, "label", label)

    return metrics_df


def metadata_exists(video_path):
    return os.path.exists(metadata_path_for_video(video_path))


def aggregate_video_metrics(frame_metrics):
    metric_cols = [
        col for col in frame_metrics.columns
        if col.startswith("sift_") or "_sift_" in col or col.startswith("patch_") or "_patch_" in col
    ]

    agg = frame_metrics[metric_cols].agg(["mean", "std", "median"])
    values = {}

    for metric in metric_cols:
        values[f"{metric}_mean"] = agg.loc["mean", metric]
        values[f"{metric}_std"] = agg.loc["std", metric]
        values[f"{metric}_median"] = agg.loc["median", metric]

    return values


def auc_from_scores(labels, scores, positive_label="Fake"):
    valid = [(label, score) for label, score in zip(labels, scores) if pd.notna(score)]
    pos = [score for label, score in valid if label == positive_label]
    neg = [score for label, score in valid if label != positive_label]

    if len(pos) == 0 or len(neg) == 0:
        return np.nan

    wins = 0
    ties = 0

    for pos_score in pos:
        for neg_score in neg:
            wins += pos_score > neg_score
            ties += pos_score == neg_score

    return (wins + 0.5 * ties) / (len(pos) * len(neg))


def cohen_d_by_label(values, labels, positive_label="Fake"):
    values = pd.Series(values)
    labels = pd.Series(labels)

    pos = values[labels == positive_label].dropna().astype(float)
    neg = values[labels != positive_label].dropna().astype(float)

    if len(pos) < 2 or len(neg) < 2:
        return np.nan

    pooled = np.sqrt((pos.var(ddof=1) + neg.var(ddof=1)) / 2)

    if pooled == 0 or pd.isna(pooled):
        return np.nan

    return (pos.mean() - neg.mean()) / pooled


def evaluate_group_b(max_frames=120, include_patch=True):
    rows = []
    available_df = df[df['video_path'].apply(metadata_exists)].reset_index(drop=True)

    for _, row in available_df.iterrows():
        frame_metrics = all_metrics(
            row['video_path'],
            max_frames=max_frames,
            label=row['Video Ground Truth'],
            include_patch=include_patch,
        )

        if frame_metrics.empty:
            continue

        video_row = aggregate_video_metrics(frame_metrics)
        video_row["video_name"] = os.path.basename(row['video_path'])
        video_row["label"] = row['Video Ground Truth']
        video_row["n_frames"] = len(frame_metrics)
        rows.append(video_row)

    video_metrics = pd.DataFrame(rows)

    metric_cols = [
        col for col in video_metrics.columns
        if (col.startswith("sift_") or "_sift_" in col or col.startswith("patch_") or "_patch_" in col) and col.endswith("_mean")
    ]

    report_rows = []
    for metric in metric_cols:
        auc = auc_from_scores(video_metrics['label'], video_metrics[metric])
        auc_abs = max(auc, 1 - auc) if pd.notna(auc) else np.nan
        d = cohen_d_by_label(video_metrics[metric], video_metrics['label'])

        report_rows.append({
            "metric": metric,
            "auc_fake": auc,
            "auc_abs": auc_abs,
            "cohen_d_fake_minus_real": d,
            "real_mean": video_metrics.loc[video_metrics['label'] == "Real", metric].mean(),
            "fake_mean": video_metrics.loc[video_metrics['label'] == "Fake", metric].mean(),
        })

    report = pd.DataFrame(report_rows)

    if not report.empty:
        report = report.sort_values(
            ["auc_abs", "cohen_d_fake_minus_real"],
            ascending=[False, False],
            key=lambda s: s.abs() if s.name == "cohen_d_fake_minus_real" else s,
        ).reset_index(drop=True)

    print("\nVídeos avaliados:")
    print(video_metrics['label'].value_counts())

    return video_metrics, report


## Avaliação discriminativa

Executa a extração apenas nos vídeos com metadata disponível e resume uma linha por vídeo. Use `report.head(20)` para ver quais sinais estruturais separam melhor Real/Fake nesta amostra.

In [ ]:
# Rodada rapida de validacao. Aumente max_frames para estabilizar os resultados.
video_metrics, report = evaluate_group_b(max_frames=40, include_patch=True)
display(report.head(20))
